# 🌿 Starmap Workshop
## Dominant Vegetation per Park · Riverside, CA

In this notebook you will answer a spatial question that no existing dataset answers directly:

> **What is the dominant vegetation type in each park in Riverside?**

Two source datasets exist:
1. **Riverside Parks** — park polygons with name attributes
2. **Riverside Vegetation Types** — vegetation polygons with type descriptions

**Strategy:** intersect the two layers, then for each park keep the vegetation class with the largest overlap area. The result is a new GeoJSON (same polygons as Parks, plus a `dominant_vegetation` attribute) that we will index with **Starlet** and serve as **Mapbox Vector Tiles (MVT)** for browser visualisation.

---
### Workflow
| Step | What happens |
|------|--------------|
| 1 | Install dependencies |
| 2 | Set input paths |
| 3 | Load & clean the datasets |
| 4 | Intersect and compute dominant vegetation |
| 5 | Save the derived GeoJSON |
| 6 | Build the MVT tile pyramid with Starlet |
| 7 | Start the Flask tile server and open the map |

---
## Step 1 — Install Dependencies

Run this cell once. If you are using a virtual environment, activate it first.

In [ ]:
import sys

!{sys.executable} -m pip install -q starlet geopandas pyproj datasketch flask-cors mapbox-vector-tile
print("✅ All dependencies installed.")

---
## Step 2 — Set Paths

Update the variables below to match the locations of your files.

> 📂 Download the datasets from the [workshop repository](https://github.com/tarlaun/starmap_workshop)

In [ ]:
# ── Input datasets ────────────────────────────────────────────────────────────
PARKS_PATH = "Riverside_Parks.geojson"
VEG_PATH   = "Riverside_VegetationTypes.geojson"

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_GEOJSON = "Riverside_Parks_DominantVegetation.geojson"

# ── Starlet tile settings ─────────────────────────────────────────────────────
DATASET_NAME = "Parks_DominantVeg"
ZOOM         = 10      # max zoom level for MVT generation
NUM_TILES    = 40      # index tile count

print(f"Parks:      {PARKS_PATH}")
print(f"Vegetation: {VEG_PATH}")
print(f"Output:     {OUTPUT_GEOJSON}")

---
## Step 3 — Load & Clean the Datasets

### Why do we change the map projection?

Our source files (GeoJSON) are stored in **WGS84 (EPSG:4326)** — the familiar latitude/longitude system used by GPS and web maps. Coordinates in WGS84 are measured in **degrees**, not metres.

That matters here because the core question we are answering is: *which vegetation polygon has the biggest overlap area inside each park?* To compare areas, we need `.geometry.area` to return meaningful, comparable numbers. In WGS84, area is measured in **square degrees** — and one degree of longitude covers very different distances depending on how far you are from the equator:

```
At the equator:    1° longitude ≈ 111 km
At 34° N (Riverside): 1° longitude ≈  92 km   ← ~17% smaller
```

This distortion means area comparisons in WGS84 can be subtly wrong — enough that the *wrong* vegetation class could win for a park near the edge of the dataset.

The fix is to reproject into an **equal-area CRS** before doing any area calculation. We use **California Albers (EPSG:3310)**, a projection specifically designed to preserve area for California. In this CRS, coordinates are in **metres**, and `.geometry.area` gives accurate square metres.

```
 WGS84 (degrees)                California Albers (metres)
 ┌──────────────────┐           ┌──────────────────┐
 │  coords in °lat/lon│  →      │  coords in metres │
 │  area = □ degrees │ reproject│  area = m²  ✓     │
 └──────────────────┘           └──────────────────┘
```

**CRS strategy used in this notebook:**

| Stage | CRS | Reason |
|-------|-----|--------|
| Load from GeoJSON | WGS84 — EPSG:4326 | GeoJSON standard |
| Intersect & measure area | California Albers — EPSG:3310 | Accurate area in m² |
| Save output GeoJSON | WGS84 — EPSG:4326 | Required by Starlet and web maps |

---

### What does `buffer(0)` do?

After reprojecting, a zero-distance `buffer(0)` is applied to every geometry. This is a standard repair trick: it resolves minor topology issues such as self-intersections or duplicate vertices that can accumulate during reprojection and would otherwise cause `overlay()` to crash or produce empty results.

In [ ]:
import geopandas as gpd

CRS_EQUAL_AREA = "EPSG:3310"   # California Albers — accurate area in metres
CRS_WGS84      = "EPSG:4326"   # WGS84 — required for output / web maps

def load_and_clean(path, columns, crs):
    """Load a GeoJSON, keep requested columns, reproject, and fix geometries."""
    gdf = gpd.read_file(path)[columns + ["geometry"]]
    gdf = gdf.to_crs(crs)                                  # reproject to equal-area
    gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty].copy()
    gdf["geometry"] = gdf.geometry.buffer(0)               # repair any topology issues
    return gdf

print("Loading parks …")
parks = load_and_clean(PARKS_PATH, ["PARK_NAME"], CRS_EQUAL_AREA)

print("Loading vegetation types …")
veg = load_and_clean(VEG_PATH, ["VEGDESC"], CRS_EQUAL_AREA)

print(f"\nCRS in use: {parks.crs}")
print(f"✅ {len(parks)} parks  |  {len(veg)} vegetation polygons")
parks.head(3)

---
## Step 4 — Intersect and Compute Dominant Vegetation

**How it works:**
1. `gpd.overlay()` computes the geometric intersection of every park polygon against every vegetation polygon — each overlapping fragment becomes its own row.
2. For each fragment we calculate its area (in m², accurate because we are in EPSG:3310).
3. Per park, we keep only the row with the largest area — that vegetation class is the *dominant* one.
4. Parks with no overlap at all are labelled `Unknown`.

```
 Parks layer        Vegetation layer      After overlay()         Keep largest
 ┌──────────┐       ░░░░░│▓▓▓▓▓           ┌───────┐ ┌──────┐     ┌───────┐
 │          │   ∩   ░░░░░│▓▓▓▓▓   =       │ Oak   │ │Shrub │  →  │ Oak   │ ✓
 │  Park A  │       ░░░░░│▓▓▓▓▓           │ 8000m²│ │2000m²│     │(dom.) │
 └──────────┘       ░░░░░│▓▓▓▓▓           └───────┘ └──────┘     └───────┘
   Oak = 80%  Shrub = 20%
```

In [ ]:
print("Intersecting parks with vegetation …")
intersections = gpd.overlay(parks, veg, how="intersection", keep_geom_type=False)

if intersections.empty:
    print("⚠️  No overlaps found. All parks will be labelled 'Unknown'.")
    result = parks.copy()
    result["dominant_vegetation"] = "Unknown"
else:
    # Area is in m² because we are in EPSG:3310 (equal-area)
    intersections["overlap_area"] = intersections.geometry.area

    dominant = (
        intersections
        .sort_values(["PARK_NAME", "overlap_area"], ascending=[True, False])
        .drop_duplicates(subset=["PARK_NAME"])   # keep largest-area row per park
        [["PARK_NAME", "VEGDESC"]]
        .rename(columns={"VEGDESC": "dominant_vegetation"})
    )

    result = parks.merge(dominant, on="PARK_NAME", how="left")
    result["dominant_vegetation"] = result["dominant_vegetation"].fillna("Unknown")

result = result[["PARK_NAME", "dominant_vegetation", "geometry"]].copy()

print(f"\n✅ Done. {len(result)} parks processed.")
print("\nVegetation class breakdown:")
print(result["dominant_vegetation"].value_counts().to_string())

---
## Step 5 — Save the Derived GeoJSON

Now that the dominant vegetation has been determined, we reproject back to **WGS84 (EPSG:4326)** before saving. This is the CRS expected by Starlet, OpenLayers, and all standard web mapping tools — coordinates must be in longitude/latitude degrees.

In [ ]:
output_wgs84 = result.to_crs(CRS_WGS84)
output_wgs84.to_file(OUTPUT_GEOJSON, driver="GeoJSON")

print(f"✅ Saved → {OUTPUT_GEOJSON}")
print(f"   {len(output_wgs84)} features  |  CRS: {output_wgs84.crs}")
output_wgs84.head()

---
## Step 6 — Build the MVT Tile Pyramid with Starlet

**Starlet** indexes the GeoJSON and generates a directory of **.mvt** files (Mapbox Vector Tiles) organised by zoom / x / y — the same tile scheme used by all web maps.

| Parameter | Value | Notes |
|-----------|-------|-------|
| `zoom` | 10 | Max zoom level — appropriate for city-scale data. Can zoom in to deeper zoom levels but tiles will be generated on-the-fly by the visualization server. |
| `threshold` | 0 | Generate all MVT tiles larger than this size (in bytes). Smaller tiles will be generated on-the-fly when visualization server is running. |

> ⏳ This step may take a minute depending on the size of your dataset.

In [ ]:
import starlet
from pathlib import Path

DATASET_ROOT = Path("./datasets")
OUTDIR       = DATASET_ROOT / DATASET_NAME
DATASET_ROOT.mkdir(parents=True, exist_ok=True)
ZOOM = 10
threshold = 0

print(f"Building tile pyramid for '{DATASET_NAME}' …")

tile_result, mvt_result = starlet.build(
    input     = OUTPUT_GEOJSON,
    outdir    = str(OUTDIR),
    zoom      = ZOOM,
    threshold = THRESHOLD,
)

print(f"\n✅ Tile index built")
print(f"   rows:        {tile_result.total_rows}")
print(f"   bbox:        {tile_result.bbox}")
print(f"\n✅ MVT tiles generated")
print(f"   zoom levels: {mvt_result.zoom_levels}")
print(f"   tile count:  {mvt_result.tile_count}")
print(f"   output dir:  {mvt_result.outdir}")

---
## Step 7 — Start the Tile Server and Open the Map

The cell below starts a local **Flask** server that:
- serves `map.html` at `http://127.0.0.1:8765/`
- serves tiles at `/tiles/<dataset>/mvt/<z>/<x>/<y>.mvt`

Then it opens the map in your default browser.

> ⚠️ **`map.html` must be in the same directory as this notebook.**  
> The server runs in a background thread — **restart the kernel** to stop it.

In [ ]:
import threading
import time
import webbrowser
import warnings
from flask import Flask, Response, abort, send_file

warnings.filterwarnings("ignore", category=FutureWarning)

HOST = "127.0.0.1"
PORT = 8765

map_file = Path("map.html")
if not map_file.exists():
    raise FileNotFoundError(
        "map.html not found. Make sure it is in the same directory as this notebook."
    )

app = Flask(__name__)

@app.route("/")
def index():
    return Response(map_file.read_text(encoding="utf-8"), mimetype="text/html")

@app.route("/datasets/<dataset>/mvt/<int:z>/<int:x>/<int:y>.mvt")
def serve_mvt(dataset, z, x, y):
    tile_path = DATASET_ROOT / dataset / "mvt" / str(z) / str(x) / f"{y}.mvt"
    if not tile_path.exists():
        abort(404)
    return send_file(tile_path, mimetype="application/vnd.mapbox-vector-tile", conditional=True)

thread = threading.Thread(
    target=lambda: app.run(host=HOST, port=PORT, debug=False, use_reloader=False),
    daemon=True
)
thread.start()
time.sleep(2)

url = f"http://{HOST}:{PORT}/?dataset={DATASET_NAME}"
print(f"✅ Server running at {url}")
print("   Tiles served from:", DATASET_ROOT)
print("\n🗺️  Opening map in browser …")
webbrowser.open(url)
print("\nTo stop the server, restart the kernel (Kernel → Restart).")

---
## Notes & Troubleshooting

**Map is blank / tiles not loading**  
Check that the `datasets/` folder was created and contains `.mvt` files. Re-run Step 6.

**`Address already in use` error**  
Port 8765 is still occupied from a previous run. Restart the kernel, or change `PORT` to another value (e.g. `8766`) and re-run Steps 6 and 7.

**Slow intersection**  
Large or complex vegetation polygons slow down `gpd.overlay()`. Try simplifying the vegetation layer first:
```python
veg["geometry"] = veg.geometry.simplify(5)   # 5 metres tolerance in EPSG:3310
```

**Wrong vegetation class assigned to a park**  
This is the most common symptom of running the intersection in WGS84 instead of an equal-area CRS. Confirm that `CRS in use: EPSG:3310` appeared in the Step 3 output before proceeding.

---
📦 **Workshop resources:** [github.com/tarlaun/starmap_workshop](https://github.com/tarlaun/starmap_workshop)  
📖 **Starlet docs:** [pypi.org/project/starlet](https://pypi.org/project/starlet/)